<a href="https://colab.research.google.com/github/NataliiaFakas/TFG_BMW/blob/main/6_RR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ridge Regression


---


## 1. Cargar los datos

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

ruta_macro = "/content/drive/MyDrive/TFG_BMW/datasets/bmw_dataset_variables_macro.csv"
ruta_bmw = "/content/drive/MyDrive/TFG_BMW/datasets/ventas_bmw.csv"

df_macro = pd.read_csv(ruta_macro)
df_bmw = pd.read_csv(ruta_bmw)

display(df_macro.head())
display(df_bmw.head())

Mounted at /content/drive


,Year,Unemployment_Rate,GDP_Growth,Deposit_Facility,HICP,Industrial_Production,Retail_Sales_Growth,Consumer_Confidence,Compensation_Per_Employee,Brent_Oil_Price,EUR_USD,Euribor_12M,Population_Total
0,2005,8.9,1.7,1.25,2.18,153.2,0.03,-4.2,31.22,54.57,1.2441,2.19,434585887
1,2006,8.2,3.2,2.50,2.19,147.7,0.04,-3.5,31.95,65.16,1.2556,3.08,436041018
2,2007,7.2,2.9,3.00,2.13,145.5,0.03,-2.1,32.75,72.44,1.3705,4.24,437514477
3,2008,7.1,0.4,2.00,3.30,148.0,0.01,-10.8,33.86,96.94,1.4708,4.63,439074280
4,2009,9.0,-4.4,0.25,0.29,144.5,-0.03,-21.7,34.41,61.74,1.3948,1.31,440467898


,Año,Ventas_BMW_Unidades
0,2005,1126768
1,2006,1185088
2,2007,1276793
3,2008,1202239
4,2009,1068770


## 2. Seleccionar las variables macroeconómicas

In [2]:
variables_modelo = [
    "Year",
    "Compensation_Per_Employee",
    "Industrial_Production",
    "Population_Total",
    "EUR_USD"
]

df_macro_reducido = df_macro[variables_modelo]

display(df_macro_reducido.head())

,Year,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD
0,2005,31.22,153.2,434585887,1.2441
1,2006,31.95,147.7,436041018,1.2556
2,2007,32.75,145.5,437514477,1.3705
3,2008,33.86,148.0,439074280,1.4708
4,2009,34.41,144.5,440467898,1.3948


## 3. Unir las ventas de BMW con las variables macroeconómicas

In [3]:
def añadir_variables_macro(df_bmw, df_macro):

    df_final = df_bmw.merge(
        df_macro[variables_modelo],
        on="Year",
        how="left"
    )

    return df_final

In [4]:
df_bmw_filtrado = df_bmw.rename(
    columns={"Año": "Year"}
)

data = añadir_variables_macro(
    df_bmw_filtrado,
    df_macro_reducido
)

display(data.head())

print("Shape del dataset definitivo:", data.shape)

,Year,Ventas_BMW_Unidades,Compensation_Per_Employee,Industrial_Production,Population_Total,EUR_USD
0,2005,1126768,31.22,153.2,434585887,1.2441
1,2006,1185088,31.95,147.7,436041018,1.2556
2,2007,1276793,32.75,145.5,437514477,1.3705
3,2008,1202239,33.86,148.0,439074280,1.4708
4,2009,1068770,34.41,144.5,440467898,1.3948


Shape del dataset definitivo: (21, 6)


## 4. Comprobar que no existen valores perdidos

In [5]:
print("Valores nulos por variable:")
display(data.isnull().sum())

Valores nulos por variable:


,0
Year,0
Ventas_BMW_Unidades,0
Compensation_Per_Employee,0
Industrial_Production,0
Population_Total,0
EUR_USD,0


## 5. Crear la partición temporal
Aquí vamos a utilizar la misma división que en Random Forest:

- Train: 2005–2018
- Dev: 2019–2021
- Test: 2022–2025

In [6]:
train = data[data["Year"] <= 2018].copy()

dev = data[
    (data["Year"] >= 2019) &
    (data["Year"] <= 2021)
].copy()

test = data[
    data["Year"] >= 2022
].copy()

print("Observaciones Train:", len(train))
print("Observaciones Dev:", len(dev))
print("Observaciones Test:", len(test))

Observaciones Train: 14
Observaciones Dev: 3
Observaciones Test: 4


## 6. Crear X e y

In [7]:
X_train = train.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_train = train["Ventas_BMW_Unidades"]

X_dev = dev.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_dev = dev["Ventas_BMW_Unidades"]

X_test = test.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_test = test["Ventas_BMW_Unidades"]

In [8]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_dev:", X_dev.shape)
print("y_dev:", y_dev.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (14, 4)
y_train: (14,)
X_dev: (3, 4)
y_dev: (3,)
X_test: (4, 4)
y_test: (4,)


## 7. Preparar Ridge Regression

En Ridge es importante estandarizar las variables, porque las variables macroeconómicas están expresadas en escalas muy diferentes. Y para evitar errores y fugas de información, utilizamos un Pipeline.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

pipeline_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

## 8. Validación mediante TimeSeriesSplit

Utilizamos únicamente los datos disponibles hasta 2021 para seleccionar el hiperparámetro alpha.

In [11]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

RANDOM_STATE = 42

data_train_dev = data[
    data["Year"] <= 2021
].copy()

X_train_dev = data_train_dev.drop(
    columns=["Year", "Ventas_BMW_Unidades"]
)

y_train_dev = data_train_dev["Ventas_BMW_Unidades"]

tscv = TimeSeriesSplit(n_splits=4)

Podemos comprobar las particiones:

In [12]:
for i, (train_index, val_index) in enumerate(
    tscv.split(X_train_dev),
    start=1
):

    print(
        f"Fold {i}: "
        f"Train = {len(train_index)} observaciones, "
        f"Validación = {len(val_index)} observaciones"
    )

Fold 1: Train = 5 observaciones, Validación = 3 observaciones
Fold 2: Train = 8 observaciones, Validación = 3 observaciones
Fold 3: Train = 11 observaciones, Validación = 3 observaciones
Fold 4: Train = 14 observaciones, Validación = 3 observaciones


## 9. Buscar el mejor valor de alpha

In [13]:
param_grid_ridge = {
    "ridge__alpha": [
        0.01,
        0.1,
        1,
        10,
        100,
        1000
    ]
}

grid_ridge = GridSearchCV(
    estimator=pipeline_ridge,
    param_grid=param_grid_ridge,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_ridge.fit(
    X_train_dev,
    y_train_dev
)

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=4, test_size=None),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('ridge', Ridge())]),
             n_jobs=-1,
             param_grid={'ridge__alpha': [0.01, 0.1, 1, 10, 100, 1000]},
             scoring='neg_root_mean_squared_error')

## 10. Ver los mejores parámetros

In [14]:
print("Mejores parámetros Ridge:")
print(grid_ridge.best_params_)

print(
    "\nMejor RMSE medio de validación:",
    -grid_ridge.best_score_
)

Mejores parámetros Ridge:
{'ridge__alpha': 1}

Mejor RMSE medio de validación: 172726.51765651908


## 11. Entrenar el Ridge definitivo

Utilizamos la configuración seleccionada y entrenamos el modelo con todos los datos de entrenamiento disponibles hasta 2021.

In [15]:
modelo_ridge = grid_ridge.best_estimator_

modelo_ridge.fit(
    X_train_dev,
    y_train_dev
)

Pipeline(steps=[('scaler', StandardScaler()), ('ridge', Ridge(alpha=1))])

## 12. Realizar las predicciones sobre el test

In [16]:
pred_test_ridge = modelo_ridge.predict(
    X_test
)
predicciones_ridge = test[
    ["Year", "Ventas_BMW_Unidades"]
].copy()

predicciones_ridge["Prediccion_Ridge"] = pred_test_ridge

display(predicciones_ridge)

,Year,Ventas_BMW_Unidades,Prediccion_Ridge
17,2022,2100692,2.445910e+06
18,2023,2253835,2.542766e+06
19,2024,2200177,2.688099e+06
20,2025,2169761,2.803533e+06


## 13. Calcular MSE y RMSE

In [17]:
from sklearn.metrics import mean_squared_error

mse_test_ridge = mean_squared_error(
    y_test,
    pred_test_ridge
)

rmse_test_ridge = np.sqrt(
    mse_test_ridge
)

print(f"MSE Ridge en test: {mse_test_ridge:,.2f}")
print(f"RMSE Ridge en test: {rmse_test_ridge:,.2f}")

MSE Ridge en test: 210,597,858,918.49
RMSE Ridge en test: 458,909.42


## 14. Calcular MAE y R²

In [18]:
from sklearn.metrics import mean_absolute_error, r2_score

mae_test_ridge = mean_absolute_error(
    y_test,
    pred_test_ridge
)

r2_test_ridge = r2_score(
    y_test,
    pred_test_ridge
)

print(f"MAE Ridge en test: {mae_test_ridge:,.2f}")
print(f"RMSE Ridge en test: {rmse_test_ridge:,.2f}")
print(f"R² Ridge en test: {r2_test_ridge:.4f}")

MAE Ridge en test: 438,960.73
RMSE Ridge en test: 458,909.42
R² Ridge en test: -67.7760


## 15. Tabla final de MAE, RMSE y R²

In [19]:
resultado_ridge = pd.DataFrame({
    "Modelo": ["Ridge Regression"],
    "MAE": [mae_test_ridge],
    "RMSE": [rmse_test_ridge],
    "R²": [r2_test_ridge]
})

display(resultado_ridge)

,Modelo,MAE,RMSE,R²
0,Ridge Regression,438960.725626,458909.423436,-67.776021
